# DREAM AI — notebook-backed chat runtime

This single connector powers **DREAM NOTEBOOK cells and DREAM AI user chat** from the same Kaggle GPU session. When the worker connects, DREAM HOST automatically loads the existing `Qwen/Qwen3-1.7B` base model and `msjnr0477/dream-ai-adapter` into the persistent chat kernel. No training starts and no saved checkpoint is replaced.

Required Kaggle Secrets: `DREAM_HOST_NOTEBOOK_WORKER_TOKEN` and `HF_TOKEN`. Optional: `DREAM_NOTEBOOK_ARTIFACT_TOKEN` for R2 notebook checkpoint/artifact operations. The chat worker token is no longer required because chat requests are routed through DREAM NOTEBOOK.

Keep this cell running while users are chatting. The model stays loaded in the notebook kernel and is reused for later requests. Do not start model training in the same GPU session while chat is active.


In [ ]:
import os
import sys
import time
import hashlib
import subprocess
from pathlib import Path
import requests
import torch
from kaggle_secrets import UserSecretsClient

HOST = "https://dream-host-live.onrender.com"
RAW = "https://raw.githubusercontent.com/mjnr0477/dream-ai/fix/notebook-backed-dream-chat/dream-host/notebook_worker.py"
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before starting DREAM AI.")

secrets = UserSecretsClient()
def read_secret(name):
    try:
        value = str(secrets.get_secret(name) or "").strip()
    except Exception as exc:
        raise RuntimeError("Missing Kaggle Secret: " + name + ". Add it under Add-ons → Secrets.") from exc
    if len(value) < 32:
        raise RuntimeError("Kaggle Secret " + name + " is missing or too short.")
    return value

def read_optional_secret(name):
    try:
        return str(secrets.get_secret(name) or "").strip()
    except Exception:
        return ""

notebook_token = read_secret("DREAM_HOST_NOTEBOOK_WORKER_TOKEN")
hf_token = read_secret("HF_TOKEN")
artifact_token = read_optional_secret("DREAM_NOTEBOOK_ARTIFACT_TOKEN")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("DREAM HOST:", HOST)
print("R2 artifact access:", "enabled" if artifact_token else "optional secret not set")

response = requests.get(RAW, timeout=60)
response.raise_for_status()
worker_file = Path("/kaggle/working/dream_notebook_worker.py")
worker_file.write_bytes(response.content)
print("Notebook worker SHA-256:", hashlib.sha256(response.content).hexdigest())

cache_root = Path("/kaggle/working/dream-hf-cache")
cache_root.mkdir(parents=True, exist_ok=True)
worker_env = os.environ.copy()
worker_env.update({
    "DREAM_HOST_URL": HOST,
    "DREAM_HOST_NOTEBOOK_WORKER_TOKEN": notebook_token,
    "DREAM_NOTEBOOK_HF_TOKEN": hf_token,
    "DREAM_HOST_NOTEBOOK_TRUSTED_MODE": "1",
    "DREAM_HOST_NOTEBOOK_KERNEL_IDLE_SECONDS": "86400",
    "HF_HOME": str(cache_root)
})
if artifact_token:
    worker_env["DREAM_NOTEBOOK_ARTIFACT_TOKEN"] = artifact_token
else:
    worker_env.pop("DREAM_NOTEBOOK_ARTIFACT_TOKEN", None)
for key in ("DREAM_HOST_WORKER_TOKEN", "HF_TOKEN"):
    worker_env.pop(key, None)

print("Starting the shared DREAM NOTEBOOK + AI chat runtime.")
print("DREAM HOST will automatically preload the saved Qwen adapter after this worker registers.")
worker_process = None
try:
    while True:
        worker_process = subprocess.Popen([sys.executable, "-u", str(worker_file)], env=worker_env)
        print("DREAM NOTEBOOK worker process started.")
        while worker_process.poll() is None:
            time.sleep(2)
        exit_code = worker_process.returncode
        print("DREAM NOTEBOOK worker exited with code", exit_code, "— restarting in 5 seconds.")
        time.sleep(5)
except KeyboardInterrupt:
    print("Stopping DREAM AI + notebook GPU runtime…")
finally:
    if worker_process is not None and worker_process.poll() is None:
        worker_process.terminate()
        try:
            worker_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            worker_process.kill()
print("DREAM AI + notebook GPU runtime stopped.")
